
# Laboratorio 4 (adaptado al español): Tools para outreach comercial con CrewAI

Este notebook adapta el laboratorio original **L4_tools_customer_outreach.ipynb** a un ejemplo completamente en español y listo para ejecutarse en **Google Colab**.

## Objetivos de aprendizaje
En este laboratorio vas a practicar cómo construir un **sistema multiagente de ventas/outreach** usando:

- **Agentes especializados**
- **Tasks bien definidas**
- **Tools reales**
- **Tools personalizadas**
- **Uso de archivos locales como conocimiento estructurado**
- **Búsqueda web con Serper**
- **Memoria y caché implícita del framework**

## Caso de estudio
Vamos a construir un crew comercial con dos agentes:

1. **Investigador de leads**
2. **Ejecutivo de outreach**

El sistema recibirá información básica de una organización objetivo y deberá:
- investigar el lead,
- entender su contexto,
- revisar guías comerciales internas,
- y redactar mensajes de contacto personalizados en español.


## 1. Instalación de librerías

In [ ]:
!pip -q install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29


## 2. Configuración inicial

Este laboratorio usa:
- `OPENAI_API_KEY`
- `SERPER_API_KEY`

Puedes ejecutar esto directamente en Colab.


In [ ]:
import os
import warnings
from getpass import getpass

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = 'sk-xxx'
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = 'xxx'
# Puedes cambiar el modelo si deseas
#os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

print("Modelo configurado:", os.environ["OPENAI_MODEL_NAME"])
print("SERPER_API_KEY cargada:", "Sí" if os.environ.get("SERPER_API_KEY") else "No")


## 3. Importaciones principales

In [ ]:
from crewai import Agent, Task, Crew
from IPython.display import Markdown, display



## 4. Preparación de conocimiento estructurado local

Vamos a simular una carpeta interna de la empresa con guías para tratar distintos tipos de clientes.


In [ ]:

os.makedirs("instrucciones_comerciales", exist_ok=True)

guia_startups = '''
Guía para startups tecnológicas:
- Enfatizar velocidad de implementación.
- Resaltar flexibilidad, prototipado rápido y capacidad de iteración.
- Hablar de automatización, eficiencia operativa y escalabilidad.
- Mantener el mensaje corto, concreto y orientado a impacto inmediato.
'''

guia_medianas = '''
Guía para empresas medianas:
- Enfatizar productividad y optimización de procesos.
- Resaltar integración con herramientas existentes.
- Hablar de reducción de costos operativos y mejora de calidad.
- Mantener un tono profesional y consultivo.
'''

guia_grandes = '''
Guía para grandes organizaciones:
- Enfatizar gobernanza, escalabilidad y robustez.
- Hablar de seguridad, trazabilidad y despliegue controlado.
- Resaltar arquitectura, integración empresarial y soporte a gran escala.
- Mantener un tono ejecutivo, sólido y orientado a negocio.
'''

with open("instrucciones_comerciales/startups.txt", "w", encoding="utf-8") as f:
    f.write(guia_startups)

with open("instrucciones_comerciales/empresas_medianas.txt", "w", encoding="utf-8") as f:
    f.write(guia_medianas)

with open("instrucciones_comerciales/grandes_organizaciones.txt", "w", encoding="utf-8") as f:
    f.write(guia_grandes)

print("Archivos creados:", os.listdir("instrucciones_comerciales"))



## 5. Herramientas (Tools)

Usaremos tools reales del ecosistema CrewAI:
- `DirectoryReadTool`
- `FileReadTool`
- `SerperDevTool`

Y además construiremos una **tool personalizada**:
- `HerramientaAnalisisSentimiento`


In [ ]:

from crewai_tools import DirectoryReadTool, FileReadTool, SerperDevTool
from crewai_tools import BaseTool

herramienta_directorio = DirectoryReadTool(directory="instrucciones_comerciales")
herramienta_archivo = FileReadTool()
herramienta_busqueda = SerperDevTool()


### 5.1 Tool personalizada

In [ ]:

class HerramientaAnalisisSentimiento(BaseTool):
    name: str = "Herramienta de Análisis de Sentimiento"
    description: str = (
        "Analiza el tono general de un texto comercial y devuelve una evaluación simple. "
        "Úsala cuando necesites validar si un mensaje es positivo, neutral o demasiado agresivo."
    )

    def _run(self, text: str) -> str:
        texto = text.lower()
        if any(p in texto for p in ["gracias", "apoyo", "colaboración", "beneficio", "mejora"]):
            return "Sentimiento estimado: positivo y colaborativo."
        if any(p in texto for p in ["urgente", "problema", "riesgo", "fallo"]):
            return "Sentimiento estimado: potencialmente tenso o negativo."
        return "Sentimiento estimado: neutral/profesional."

herramienta_sentimiento = HerramientaAnalisisSentimiento()
herramienta_sentimiento



## 6. Definición de agentes

Tendremos dos agentes:
1. **Investigador de leads**
2. **Ejecutivo de outreach**


### 6.1 Agente: Investigador de leads

In [ ]:

investigador_leads = Agent(
    role="Investigador de Leads B2B",
    goal="Identificar y perfilar clientes potenciales de alto valor que encajen con el perfil ideal",
    backstory=(
        "Trabajas en una empresa que construye sistemas multiagente con CrewAI. "
        "Tu responsabilidad es investigar organizaciones objetivo, entender su contexto, "
        "inferir su tipo de empresa y generar un perfil útil para el equipo comercial. "
        "Debes combinar investigación web con guías internas de la empresa."
    ),
    allow_delegation=False,
    verbose=True
)


### 6.2 Agente: Ejecutivo de outreach

In [ ]:

ejecutivo_outreach = Agent(
    role="Ejecutivo de Outreach Comercial",
    goal="Crear mensajes de contacto personalizados y estratégicos para cada lead",
    backstory=(
        "Eres un ejecutivo comercial experto en comunicación B2B. "
        "Tu trabajo consiste en tomar el perfil investigado del lead, identificar el mejor ángulo de contacto "
        "y redactar mensajes claros, profesionales y persuasivos en español. "
        "Debes adaptar el tono según el tipo de organización y usar un enfoque consultivo."
    ),
    allow_delegation=False,
    verbose=True
)



## 7. Definición de tareas


### 7.1 Tarea: Perfilado del lead

In [ ]:

tarea_perfilado = Task(
    description=(
        "Analiza el siguiente lead comercial:\n"
        "- Nombre de la organización: {nombre_lead}\n"
        "- Industria: {industria}\n"
        "- Persona clave: {persona_clave}\n"
        "- Cargo: {cargo}\n"
        "- Hito o contexto reciente: {hito}\n\n"
        "Tu tarea es:\n"
        "1. Investigar la organización usando búsqueda web.\n"
        "2. Inferir si se parece más a una startup, una empresa mediana o una gran organización.\n"
        "3. Revisar las guías comerciales internas disponibles en el directorio local.\n"
        "4. Elaborar un perfil estratégico del lead con recomendaciones de acercamiento."
    ),
    expected_output=(
        "Un perfil del lead en español que incluya: contexto de la organización, "
        "tipo de empresa estimado, posibles necesidades, oportunidades de contacto "
        "y recomendaciones estratégicas iniciales."
    ),
    agent=investigador_leads,
    tools=[herramienta_busqueda, herramienta_directorio, herramienta_archivo]
)


### 7.2 Tarea: Redacción del mensaje comercial

In [ ]:

tarea_outreach = Task(
    description=(
        "Usando el perfil investigado previamente del lead:\n"
        "- redacta un correo principal de contacto en español,\n"
        "- redacta una segunda variante alternativa,\n"
        "- analiza el tono del mensaje con la herramienta de sentimiento,\n"
        "- y asegúrate de que la propuesta sea pertinente para el contexto del lead.\n\n"
        "El mensaje debe estar orientado a generar interés inicial y abrir una conversación."
    ),
    expected_output=(
        "Dos versiones de mensaje de outreach en español, bien redactadas y personalizadas, "
        "más una breve observación sobre el tono del mensaje."
    ),
    agent=ejecutivo_outreach,
    tools=[herramienta_busqueda, herramienta_sentimiento]
)



## 8. Construcción del crew

Activamos `memory=True` para que el crew pueda compartir mejor el contexto entre tareas.


In [ ]:

equipo_comercial = Crew(
    agents=[investigador_leads, ejecutivo_outreach],
    tasks=[tarea_perfilado, tarea_outreach],
    verbose=2,
    memory=True
)


## 9. Ejecución del crew

In [ ]:

inputs = {
    "nombre_lead": "DeepLearning.AI",
    "industria": "Plataforma de educación en inteligencia artificial",
    "persona_clave": "Andrew Ng",
    "cargo": "Founder",
    "hito": "Expansión de programas educativos y crecimiento en formación de IA"
}

resultado = equipo_comercial.kickoff(inputs=inputs)


## 10. Visualización del resultado

In [ ]:
display(Markdown(resultado))

## 11. Probar con otro lead

In [ ]:

inputs_2 = {
    "nombre_lead": "Bancolombia",
    "industria": "Servicios financieros y banca",
    "persona_clave": "Director de Innovación",
    "cargo": "Chief Innovation Officer",
    "hito": "Interés en automatización y modernización digital"
}

resultado_2 = equipo_comercial.kickoff(inputs=inputs_2)
display(Markdown(resultado_2))



## 12. Qué conceptos ilustra este laboratorio

- **Especialización de agentes**
- **Tools reales**
- **Conocimiento estructurado local**
- **Error handling y robustez**
- **Contexto comercial**

Muestra cómo los multiagentes son muy adecuados para:
- ventas,
- lead scoring,
- outreach,
- generación de mensajes personalizados.



## 13. Extensiones sugeridas para estudiantes

### Extensión 1
Agregar un tercer agente:
- **Revisor de calidad comercial**

### Extensión 2
Agregar una tool real para:
- enviar correo,
- crear draft,
- o generar salida JSON estructurada.

### Extensión 3
Convertir este laboratorio en un mini sistema:
- **Sales + RAG + Memory**

### Extensión 4
Comparar dos configuraciones:
- tools a nivel de agente
- tools a nivel de tarea



## 14. Reflexión final

Este laboratorio muestra cómo un sistema multiagente puede actuar como un pequeño equipo comercial:

- investiga,
- consulta guías internas,
- analiza contexto,
- redacta mensajes,
- y ajusta el tono.

### Mensaje clave
Un sistema multiagente no solo genera texto:
**analiza, recupera contexto, adapta estrategia y produce comunicación personalizada**.
